# Experiment 1: the original Calibrate widget (pycalibrate) on COMPAS and on our predictions

Not part of run_all.sh; it needs an extra install (below).

This opens the published tool (Xenopoulos et al. 2023, https://github.com/VIDA-NYU/pycalibrate) to
(1) repeat the paper's COMPAS exploration by hand (brush the age feature, compare curves), and
(2) look at our own patent predictions in the original tool.

Setup (once): `pip install notebook git+https://github.com/VIDA-NYU/pycalibrate` then `jupyter notebook` (classic notebook;
the widget uses notebook JS callbacks and may not render in JupyterLab or VS Code).

Two compatibility fixes:
* newer IPython moved `display` out of `IPython.core.display`; the first cell puts it back before importing pycalibrate;
* newer interpret removed `binning="uniform"`; the learned curve inside the widget may fail on interpret >= 0.3. Our script
  `e1_calibrate_repro.py` computes the same curve with `feature_types=["uniform"]` (vml/learned.py).

In [ ]:
# compatibility shim for notebookjs/pycalibrate on IPython >= 8
import IPython.core.display as _d
from IPython.display import display, HTML, Javascript
for _n, _o in (("display", display), ("HTML", HTML), ("Javascript", Javascript)):
    if not hasattr(_d, _n):
        setattr(_d, _n, _o)
import numpy as np, pandas as pd
from pycalibrate import Calibrate

In [ ]:
# COMPAS exactly as in e1_calibrate_repro.py: 50/50 split, no age, default random forest
import sys; sys.path.insert(0, '../..')
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
d = pd.read_csv('../data/compas-scores-two-years.csv')
d = d[(d.days_b_screening_arrest <= 30) & (d.days_b_screening_arrest >= -30) & (d.is_recid != -1)
      & (d.c_charge_degree != 'O') & (d.score_text != 'N/A')].copy()
d['sex_male'] = (d.sex == 'Male').astype(int); d['felony'] = (d.c_charge_degree == 'F').astype(int)
feats = ['sex_male', 'felony', 'juv_misd_count', 'juv_fel_count', 'priors_count']
tr, te = train_test_split(d, test_size=0.5, random_state=0)
rf = RandomForestClassifier(random_state=0).fit(tr[feats], tr.two_year_recid)
P = rf.predict_proba(te[feats])
Y = np.eye(2)[te.two_year_recid.to_numpy()]
cal = Calibrate(te[feats + ['age']].reset_index(drop=True))   # age is shown as a feature to brush, not used by the model
cal.add_model(P, Y, 'RF without age')
cal.visualize()

**What to do:** in the feature view brush `age` from 45 to the maximum and compare the new curve with the full
population (paper: older defendants are overpredicted when age is not a feature).

In [ ]:
# our own predictions in the original tool (test block, confirmed examiners, LR profile off)
import os
from vml.paths import stage_dir
p = pd.read_csv(stage_dir('models') / 'predictions.csv')
p = p[(p.block == 'test') & (p.primary == 1)].reset_index(drop=True)
P2 = np.column_stack([1 - p.lr_off, p.lr_off]); Y2 = np.eye(2)[p.y.to_numpy()]
cal2 = Calibrate(p[['round_n', 'prof_n', 'g101', 'g102', 'g103', 'g112', 'filing_year']])
cal2.add_model(P2, Y2, 'LR profile off')
cal2.visualize()